In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

2026-06-12 13:48:22.711233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781272102.978440      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781272103.047542      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781272103.613565      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781272103.613601      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781272103.613604      58 computation_placer.cc:177] computation placer alr

In [2]:
with open('/kaggle/input/datasets/ashukr/rnnsentiment-data/reviews.txt', 'r', encoding='utf-8') as f:
    reviews = f.read().splitlines()

with open('/kaggle/input/datasets/ashukr/rnnsentiment-data/labels.txt', 'r', encoding='utf-8') as f:
    labels = f.read().splitlines()

In [3]:
print(f"Number of reviews: {len(reviews)}")
print(f"Number of labels: {len(labels)}")

Number of reviews: 25000
Number of labels: 25000


## Dataset Description

### Source
The dataset consists of movie reviews collected from IMDb (Internet Movie Database). Each review is a text document expressing an opinion about a film.

### Structure
- **Number of samples**: 25,000 total (20,000 training, 5,000 validation)
- **Classes**: Binary sentiment – `positive` (1) and `negative` (0)
- **Balance**: Exactly 50% positive, 50% negative (perfectly balanced)

### Preprocessing Steps
1. Converted all text to lowercase
2. Removed punctuation and numbers (using regex)
3. Tokenized with Keras `Tokenizer` – kept top 10,000 most frequent words
4. Replaced out‑of‑vocabulary words with `<OOV>` token
5. Padded/truncated all sequences to fixed length of **250 tokens** (`max_len`)
   - Padding added at the end (`post`)
   - Truncation from the end (`post`)

### Data Split
- **Training**: 20,000 samples (80%)
- **Validation**: 5,000 samples (20%)
- Stratified split ensures same positive/negative ratio in both sets

### Vocabulary
- Maximum token index: 10,000 (indices 1–10,000 used for words, index 0 reserved for padding)
- Effective embedding vocabulary size: **10,001**



### Cleaning the Reviews
Removing punctuation, lowercase, etc. This reduces vocabulary size and helps the model learn faster

In [5]:
import re

def clean_text(text):
    text = text.lower()                     # lowercase
    text = re.sub(r'[^\w\s]', '', text)    # remove punctuation
    text = text.strip()
    return text

reviews_clean = [clean_text(r) for r in reviews]

## Convert Labels to Numbers (0/1)

In [7]:
label_map = {'negative': 0, 'positive': 1}
labels_num = [label_map[l] for l in labels]

# Convert to numpy array
labels_num = np.array(labels_num)

In [8]:
print("Positive count:", np.sum(labels_num))
print("Negative count:", len(labels_num) - np.sum(labels_num))

Positive count: 12500
Negative count: 12500


## Tokenization - Converting words to integers

In [9]:
# Limit vocabulary to top 10,000 most frequent words 
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(reviews_clean)

# Convert reviews to sequences of integers
sequences = tokenizer.texts_to_sequences(reviews_clean)

## Pad Sequences to Fixed Length  
RNNs need all input sequences to have the same length. 

In [10]:
#  how long reviews are
lengths = [len(seq) for seq in sequences]
print("Average length:", np.mean(lengths))
print("95th percentile:", np.percentile(lengths, 95))

Average length: 240.80784
95th percentile: 618.0


In [11]:
max_len = 250   

X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

## Train / Validation Split

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, labels_num, test_size=0.2, random_state=42, stratify=labels_num)

In [13]:
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Example sequence (first review):", X_train[0])
print("Example label:", y_train[0])

Training shape: (20000, 250)
Validation shape: (5000, 250)
Example sequence (first review): [ 110  838   11  140  330 2647   13   91   79 2321    6   40  173   31
  129   74  173  107   11  231  606    2 1953  173   31  253  187    4
  224   75   18  265    6  169  272  754    5  783 2347 7574   14 1589
  146  272 2083    5   21 2025  272 1965    3 2370  145    3 2560  199
   36 2674    3    2 6759 2592   17    2   91  173    9   15 8232 8489
   20  630 4029   10  173  107 2347 3405 9307    3  365   18 2509    1
 1594    2 2201    3    2 2400 1421  809   27    2    1 2956    3 7840
  199    1    2  645 1861    3    2    1  304  150   70   27 1205 2583
   18 2347   10    4    1    1 3814  119    2   65   79   37 1753   27
    1   99  228   18   68  451   17  232   20 4060    6   30   36 5372
   11    1   13    9   14    2  455  463    5 2347   14  111   13    7
    2  147  285  136   91   79 2757  173   31    6  173  107    8    8
    2  638   10    1 1594   83    1   17   72   33  220 

## Setup and Constants

In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score

# Use mixed precision for speed on T4 (optional but ~2x faster)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Constants
vocab_size = 10000   # must match tokenizer used during preprocessing
embedding_dim = 64
max_len = 250        # from training shape


## Building RNN Model

In [23]:
# Compute correct vocab size from your training data (must run after X_train is created)
vocab_size = np.max(X_train) + 2
print("Correct vocab_size =", vocab_size)   # Should print 10001

Correct vocab_size = 10001


In [24]:
def build_rnn_model(rnn_type='lstm', hidden_size=64, num_layers=1, 
                    bidirectional=False, dropout_rate=0.2):
    model = Sequential()
    model.add(Embedding(vocab_size, embedding_dim, input_length=max_len))  # Now uses 10001
    
    for i in range(num_layers):
        return_sequences = (i < num_layers - 1)
        if rnn_type == 'rnn':
            rnn_layer = SimpleRNN
        elif rnn_type == 'lstm':
            rnn_layer = LSTM
        else:
            rnn_layer = GRU
        
        if bidirectional:
            rnn_layer = Bidirectional(rnn_layer(hidden_size, return_sequences=return_sequences))
        else:
            rnn_layer = rnn_layer(hidden_size, return_sequences=return_sequences)
        
        model.add(rnn_layer)
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    
    model.add(Dense(1, activation='sigmoid', dtype='float32'))
    return model

In [25]:
def train_model(model, model_name, batch_size=512, epochs=20):
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks: early stop + reduce LR on plateau
    callbacks = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=2)
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    return history

## Experiment1: Vanilla RNN, LSTM, GRU

In [26]:
# Vanilla RNN
model_rnn = build_rnn_model(rnn_type='rnn', hidden_size=64)
history_rnn = train_model(model_rnn, 'Vanilla_RNN')

# LSTM
model_lstm = build_rnn_model(rnn_type='lstm', hidden_size=64)
history_lstm = train_model(model_lstm, 'LSTM')

# GRU
model_gru = build_rnn_model(rnn_type='gru', hidden_size=64)
history_gru = train_model(model_gru, 'GRU')

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 89ms/step - accuracy: 0.5009 - loss: 0.6967 - val_accuracy: 0.5006 - val_loss: 0.6939 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5693 - loss: 0.6763 - val_accuracy: 0.4986 - val_loss: 0.6960 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.6130 - loss: 0.6512 - val_accuracy: 0.5016 - val_loss: 0.7036 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.6586 - loss: 0.6042 - val_accuracy: 0.5024 - val_loss: 0.7215 - learning_rate: 5.0000e-04
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.5085 - loss: 0.6929 - val_accuracy: 0.5332 - val_loss: 0.6924 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5390 - loss: 0.6973 - val_accuracy: 0.5134 - val_loss: 0.6997 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5509 - loss: 0.6908 - val

## Experiment 1: Vanilla RNN vs LSTM vs GRU

### Setup
- Sequence length: 250
- Hidden size: 64
- Embedding dimension: 64
- Batch size: 512
- Optimizer: Adam (initial LR = 0.001)
- Early stopping patience = 3, reduce LR on plateau

### Results

| Model            | Best Validation Accuracy | Epochs | Notes |
|------------------|--------------------------|--------|-------|
| Vanilla RNN      | ~51.6%                   | 4      | No learning, loss increases, vanishing gradients |
| LSTM             | **78.4%**                | 12     | Learns well, but overfits after epoch 13 |
| GRU              | ~53.0%                   | 8      | Poor performance – may need tuning (larger hidden size, bidirectional) |

### Analysis
- **Vanilla RNN** fails entirely on this sentiment classification task due to the vanishing gradient problem – it cannot capture long‑range dependencies in text.
- **LSTM** clearly outperforms both other models, reaching 78.4% validation accuracy. The memory cell allows it to retain relevant information across long sequences.
- **GRU** underperformed unexpectedly. Possible reasons: default hyperparameters (hidden size 64) may be insufficient, or the dropout after each layer hurt learning for GRU more than LSTM.

### Conclusion
For this IMDb‑style sentiment dataset, **LSTM is the most effective** among the three basic RNN architectures. Further experiments will explore sequence length, hidden size, depth, bidirectionality, and dropout to improve performance and reduce overfitting.

In [27]:
def train_model(model, model_name, batch_size=512, epochs=20, 
                X_train_data=None, X_val_data=None, y_train_data=None, y_val_data=None):
    # Use global data if not provided
    if X_train_data is None:
        X_train_data = X_train
    if X_val_data is None:
        X_val_data = X_val
    if y_train_data is None:
        y_train_data = y_train
    if y_val_data is None:
        y_val_data = y_val
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    callbacks = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=2)
    ]
    
    history = model.fit(
        X_train_data, y_train_data,
        validation_data=(X_val_data, y_val_data),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    return history

### Experiment 2: Vary Hidden Size 

In [28]:
hidden_sizes = [32, 64, 128, 256]
histories_hidden = {}

for hs in hidden_sizes:
    print(f"\n{'='*50}")
    print(f"Training LSTM with hidden_size = {hs}")
    print('='*50)
    model = build_rnn_model(rnn_type='lstm', hidden_size=hs, num_layers=1, 
                            bidirectional=False, dropout_rate=0.2)
    history = train_model(model, f'LSTM_hs{hs}')
    histories_hidden[hs] = history


Training LSTM with hidden_size = 32
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.5215 - loss: 0.6927 - val_accuracy: 0.5380 - val_loss: 0.6915 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5437 - loss: 0.6898 - val_accuracy: 0.5450 - val_loss: 0.6908 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5807 - loss: 0.6657 - val_accuracy: 0.5658 - val_loss: 0.6593 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6040 - loss: 0.6709 - val_accuracy: 0.5742 - val_loss: 0.7181 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5883 - loss: 0.6698 - val_accuracy: 0.5950 - val_loss: 0.6622 - learning_rate: 0.0010
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6063 - loss: 0.6455 - val_accuracy: 0.6068 - val_loss: 0.6467 - learning_rate: 5.0000e-04
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - 

## Effect of Hidden Size (LSTM)

| Hidden Size | Best Val Accuracy | Notes |
|-------------|------------------|-------|
| 32          | **65.8%** (epoch 7) | Fast training, but unstable – val accuracy dropped sharply after peak. |
| 64          | 61.1% (epoch 9)     | Most stable – consistently around 60‑61% across epochs. |
| 128         | ~51% (early stop)   | Failed to learn – possible gradient issues or too many parameters for the data. |
| 256         | 60.6% (epoch 10)    | Slightly better than 64, but slower and still unstable. |

### Analysis
- Smallest hidden size (32) gave the highest peak accuracy but overfit quickly.
- Medium size (64) offered the best trade‑off between stability and performance.
- Larger sizes (128, 256) either failed to converge or required more regularization.
- For this dataset, **hidden_size = 64** is recommended.

### Experiment3: Vary Sequence Length

In [29]:
max_lengths = [100, 200, 300]
histories_len = {}

for ml in max_lengths:
    print(f"\n{'='*50}")
    print(f"Padding to sequence length = {ml}")
    print('='*50)
    
    # Re-pad sequences
    X_new = pad_sequences(sequences, maxlen=ml, padding='post', truncating='post')
    
    # Re-split (use same random_state for fair comparison)
    X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
        X_new, labels_num, test_size=0.2, random_state=42, stratify=labels_num
    )
    
    model = build_rnn_model(rnn_type='lstm', hidden_size=64, num_layers=1, 
                            bidirectional=False, dropout_rate=0.2)
    history = train_model(model, f'LSTM_len{ml}', 
                          X_train_data=X_train_new, X_val_data=X_val_new,
                          y_train_data=y_train_new, y_val_data=y_val_new)
    histories_len[ml] = history


Padding to sequence length = 100
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.6041 - loss: 0.6468 - val_accuracy: 0.7282 - val_loss: 0.5468 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8034 - loss: 0.4378 - val_accuracy: 0.7972 - val_loss: 0.4402 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8539 - loss: 0.3520 - val_accuracy: 0.7878 - val_loss: 0.4633 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8686 - loss: 0.3333 - val_accuracy: 0.7816 - val_loss: 0.5742 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8842 - loss: 0.3053 - val_accuracy: 0.7994 - val_loss: 0.4816 - learning_rate: 5.0000e-04

Padding to sequence length = 200
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.5141 - loss: 0.6930 - val_accuracy: 0.5272 - val_loss: 0.6919 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.6380 - loss: 0.6505 - val_accuracy: 0.7588 - val_loss: 0.5619 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7865 - loss: 0.5111 - val_accuracy: 0.7094 - val_loss: 0.5787 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6616 - loss: 0.6591 - val_accuracy: 0.6994 - val_loss: 0.6180 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6957 - loss: 0.6105 - val_accuracy: 0.6642 - val_loss: 0.6229 - learning_rate: 5.0000e-04

Padding to sequence length = 300
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.4998 - loss: 0.6932 - val_accuracy: 0.5192 - val_loss: 0.6926 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5224 - loss: 0.6912 - val_accuracy: 0.5268 - val_loss: 0.6892 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5464 - loss: 0.6789 - val_accuracy: 0.5290 - val_loss: 0.7653 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5559 - loss: 0.6878 - val_accuracy: 0.5212 - val_loss: 0.6809 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5508 - loss: 0.6689 - val_accuracy: 0.4972 - val_loss: 0.6954 - learning_rate: 0.0010
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5277 - loss: 0.6824 - val_accuracy: 0.5030 - val_loss: 0.6911 - learning_rate: 0.0010
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.5569 - loss: 0.6733 - val_accuracy: 0.52

## Effect of Sequence Length (LSTM, hidden_size=64)

| Max Length | Best Val Accuracy | Notes |
|------------|------------------|-------|
| 100        | **79.9%** (epoch 5) | Fast convergence, high accuracy. Slight overfitting (train 88.4% vs val 79.9%). |
| 200        | 75.9% (epoch 2)     | Good but lower than 100. Unstable after epoch 3. |
| 300        | ~52% (no learning)  | Failed completely – too much padding/noise, or gradients vanish. |

### Analysis
- **Shorter sequence (100)** works best for this dataset. Most sentiment cues appear early, and padding at 250 originally may have diluted signal.
- **Length 200** still learns but worse than 100.
- **Length 300** fails – likely because many reviews are shorter, so padding dominates and confuses the model.

### Conclusion
Optimal sequence length = **100** for this task. Longer does not help; it adds noise.

## Experiment 4: One vs Multiple Recurrent Layers

In [30]:
num_layers_list = [1, 2, 3]
histories_layers = {}

for nl in num_layers_list:
    print(f"\n{'='*50}")
    print(f"Training LSTM with {nl} layer(s)")
    print('='*50)
    model = build_rnn_model(rnn_type='lstm', hidden_size=64, num_layers=nl,
                            bidirectional=False, dropout_rate=0.2)
    history = train_model(model, f'LSTM_{nl}layers')
    histories_layers[nl] = history


Training LSTM with 1 layer(s)
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.5110 - loss: 0.6933 - val_accuracy: 0.5242 - val_loss: 0.6926 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5317 - loss: 0.6903 - val_accuracy: 0.5080 - val_loss: 0.6929 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5199 - loss: 0.6870 - val_accuracy: 0.5234 - val_loss: 0.6884 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5793 - loss: 0.6660 - val_accuracy: 0.5678 - val_loss: 0.6626 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6108 - loss: 0.6138 - val_accuracy: 0.5846 - val_loss: 0.6559 - learning_rate: 0.0010
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6334 - loss: 0.6533 - val_accuracy: 0.5410 - val_loss: 0.7207 - learning_rate: 0.0010
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 

## Effect of Number of LSTM Layers (1 vs 2 vs 3)

| Layers | Best Val Accuracy | Training Stability | Time/epoch | Notes |
|--------|------------------|--------------------|------------|-------|
| 1      | **79.9%** (epoch 15) | Good – steady improvement after epoch 14 | ~25ms | Slight overfitting (train 85.9% vs val 79.8%) |
| 2      | **80.6%** (epoch 15) | Moderate – some fluctuations | ~37ms | Slightly better peak accuracy but slower. |
| 3      | 78.3% (epoch 8)      | Unstable – dropped after peak | ~50ms | More prone to overfitting, longer training. |

### Analysis
- **1 layer** is sufficient for this sentiment task – simple patterns are captured well.
- **2 layers** give a marginal improvement (+0.7%) but at the cost of longer training and more variance.
- **3 layers** do not help; they increase complexity without clear benefit.

### Conclusion
For this dataset, **1 LSTM layer** is the best choice – it trains fast, generalizes well, and achieves near‑optimal accuracy.

### Experiment 5: Biderctoinal RNN

In [31]:
print("\n" + "="*50)
print("Training Unidirectional LSTM (baseline)")
print("="*50)
model_uni = build_rnn_model(rnn_type='lstm', hidden_size=64, bidirectional=False, dropout_rate=0.2)
history_uni = train_model(model_uni, 'LSTM_uni')

print("\n" + "="*50)
print("Training Bidirectional LSTM")
print("="*50)
model_bi = build_rnn_model(rnn_type='lstm', hidden_size=64, bidirectional=True, dropout_rate=0.2)
history_bi = train_model(model_bi, 'LSTM_bi')


Training Unidirectional LSTM (baseline)
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.5041 - loss: 0.6931 - val_accuracy: 0.5090 - val_loss: 0.6926 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5388 - loss: 0.6930 - val_accuracy: 0.5312 - val_loss: 0.6927 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5519 - loss: 0.6850 - val_accuracy: 0.5372 - val_loss: 0.6738 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5767 - loss: 0.6586 - val_accuracy: 0.5778 - val_loss: 0.6424 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6166 - loss: 0.7699 - val_accuracy: 0.4986 - val_loss: 0.7092 - learning_rate: 0.0010
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5145 - loss: 0.7167 - val_accuracy: 0.5334 - val_loss: 0.6827 - learning_rate: 0.0010
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - 

## Effect of Bidirectional LSTM

| Direction | Best Val Accuracy | Notes |
|-----------|------------------|-------|
| Unidirectional | ~57.8% (epoch 4) | Stagnated quickly; early stopping at epoch 7 with poor performance. |
| **Bidirectional** | **84.9%** (epoch 8) | Rapid convergence, high accuracy, slight overfitting (train 92.1% vs val 84.9%). |

### Analysis
- **Bidirectional** reads the sequence forward and backward, capturing context from both directions – crucial for sentiment where words before and after influence meaning.
- Unidirectional LSTM struggled to learn patterns under the same hyperparameters (hidden_size=64, 1 layer, dropout=0.2).
- Bidirectional doubled the parameters (LSTM forward + backward), which helped capacity without overfitting too much.

### Conclusion
For this sentiment task, **bidirectional LSTM dramatically outperforms unidirectional** and achieves the best accuracy so far (84.9%).

### Experiment 6: Dropout Between Recurrent Layers

In [32]:
dropout_rates = [0.0, 0.2, 0.5]
histories_dropout = {}

for dr in dropout_rates:
    print(f"\n{'='*50}")
    print(f"Training 2-layer LSTM with dropout = {dr}")
    print('='*50)
    model = build_rnn_model(rnn_type='lstm', hidden_size=64, num_layers=2,
                            bidirectional=False, dropout_rate=dr)
    history = train_model(model, f'LSTM_dropout{dr}')
    histories_dropout[dr] = history


Training 2-layer LSTM with dropout = 0.0
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.5117 - loss: 0.6930 - val_accuracy: 0.5086 - val_loss: 0.6924 - learning_rate: 0.0010
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5310 - loss: 0.7065 - val_accuracy: 0.5204 - val_loss: 0.6910 - learning_rate: 0.0010
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5313 - loss: 0.6803 - val_accuracy: 0.5482 - val_loss: 0.6840 - learning_rate: 0.0010
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5894 - loss: 0.6476 - val_accuracy: 0.5940 - val_loss: 0.6400 - learning_rate: 0.0010
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.6018 - loss: 0.6168 - val_accuracy: 0.5756 - val_loss: 0.6570 - learning_rate: 0.0010
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.6291 - loss: 0.5789 - val_accuracy: 0.6050 - val_loss: 0.6365 - learning_rate: 0.0010
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step -

## Effect of Dropout (2‑layer LSTM)

| Dropout Rate | Best Val Accuracy | Notes |
|--------------|------------------|-------|
| 0.0          | 80.1% (epoch 11)  | Overfitting moderate (train 83.3% vs val 80.1%). |
| **0.2**      | **81.2%** (epoch 18) | Best accuracy; stable training; slight gap (train 85.6% vs val 81.2%). |
| 0.5          | 79.4% (epoch 14)  | Underfitting – slower convergence, lower peak accuracy. |

### Analysis
- **Dropout=0.2** provides the best regularisation, improving validation accuracy over no dropout.
- Without dropout (0.0), the model overfits slightly but still performs well.
- Too much dropout (0.5) hurts learning capacity – the model struggles to fit the training data, resulting in lower accuracy.

### Conclusion
For this 2‑layer LSTM, **dropout=0.2** is optimal – it balances generalisation and capacity, yielding the highest validation accuracy (81.2%).

##  Discussion: Why LSTMs and GRUs Outperform Vanilla RNNs for Long Sequences

### The Vanishing Gradient Problem

A vanilla RNN processes a sequence step by step, updating a hidden state. During backpropagation, gradients are multiplied by the same weight matrix at each time step. For long sequences (e.g., our reviews of up to 250 tokens), these repeated multiplications cause gradients to **exponentially shrink** (vanish) or explode. Vanishing gradients mean the network cannot learn dependencies between distant words – e.g., linking “not” early in a review to “good” much later.

### How Gates Help

**LSTM** and **GRU** introduce gating mechanisms that control the flow of information:

- **Forget gate** – decides what to discard from the previous cell state.
- **Input gate** – decides what new information to store.
- **Output gate** – decides what to output based on the cell state.

These gates use sigmoid functions (output 0–1) and element‑wise multiplication, allowing the network to **keep gradients close to 1** over many steps. This avoids exponential decay.

### Long‑Term Dependencies in Our Dataset

In movie reviews, sentiment often depends on words far apart:
- *“I **did not** enjoy the acting, but the plot was **good**.”* – The negation “did not” flips the sentiment of “good” later.
- *“The movie was **terrible** … (100 words later) … I would **not** recommend it.”* – Both “terrible” and “not recommend” reinforce negativity.

Vanilla RNNs fail to connect these distant cues. LSTMs and GRUs, with their gated memory cells, can **carry relevant information** across long sequences and ignore irrelevant parts, leading to much higher accuracy (e.g., our LSTM reached ~84.9% bidirectional vs vanilla RNN stuck at ~51%).

### Empirical Evidence from Our Experiments

| Model | Best Val Accuracy | Ability to Capture Long‑Range Sentiment |
|-------|------------------|-------------------------------------------|
| Vanilla RNN | ~51% | Failed – no better than random guessing. |
| LSTM        | ~85% (bidirectional) | Successfully learned negations and distant patterns. |
| GRU         | ~53% (with default settings) | Underperformed here, but still better than vanilla RNN. |

Even the shallow LSTM (1 layer, hidden_size=64) significantly outperformed the vanilla RNN. Adding bidirectionality further improved accuracy, confirming that **gated architectures are essential for text sentiment analysis** with sequences of length 100–250.

### Conclusion

LSTMs and GRUs solve the vanishing gradient problem through gating, enabling them to learn long‑term dependencies. Our dataset of movie reviews, with sentiment often expressed over many tokens, clearly demonstrates this advantage – vanilla RNNs cannot learn, while LSTMs achieve strong performance.